In [22]:
#imported ufc prediction excel file after etl with power bi and copying table in power bi to excel file step 1
import os
import pandas as pd
print(os.getcwd())

# best method for paths r'personal-projects\Untitled spreadsheet (1).xlsx'
df2 = pd.read_excel(r'personal-projects\Untitled spreadsheet (2).xlsx')

def map_winner_color(row):
    winner = str(row['Winner']).strip().lower()
    fighter_a = str(row['fighter_A']).strip().lower()
    fighter_b = str(row['fighter_B']).strip().lower()

    if winner == 'nc':
        return 'nc'
    elif winner == 'd' or winner == 'draw':
        return 'd'
    elif winner == fighter_a:
        return 'red'   # Fighter A is red corner
    elif winner == fighter_b:
        return 'blue'  # Fighter B is blue corner
    else:
        return 'unknown'  # winner name doesn't match either fighter or special case

# Apply the function to create a new column, e.g. 'Winner_color'
df2['Winner_color'] = df2.apply(map_winner_color, axis=1)

print(df2[['fighter_A', 'fighter_B', 'Winner', 'Winner_color']])

C:\Users\admin\Desktop\newwine\imback
                fighter_A               fighter_B         Winner Winner_color
0           Dustin Jacoby       Kennedy Nzechukwu  Dustin Jacoby          red
1           Derrick Lewis  Marcos Rogerio de Lima  Derrick Lewis          red
2            Tom Aspinall           Marcin Tybura   Tom Aspinall          red
3           Robbie Lawler              Niko Price  Robbie Lawler          red
4           Manuel Torres           Nikolas Motta  Manuel Torres          red
...                   ...                     ...            ...          ...
8298     Alonzo Menifield                Oumar Sy  Nate Landwehr      unknown
8299        Jhonata Diniz             Alvin Hines  Ateba Gautier      unknown
8300  Daria Zhelezniakova         Melissa Mullins   Luana Santos      unknown
8301          Dhiego Lima            Jesse Taylor    Jordan Mein      unknown
8302      Gabriel Benitez         Enrique Barzola    Darren Till      unknown

[8303 rows x 4 columns]


In [23]:
df2=df2[df2['Winner_color'] != 'unknown']
print(df2['Winner_color'].value_counts())

Winner_color
red     228
nc       87
d        60
blue      7
Name: count, dtype: int64


In [24]:
#find best model step 2 create train and test splits
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

#turn results into red,blue, nc, draw
#find best model

# Assume df has 'fighter_A', 'fighter_B', fight stats columns, and 'Outcome' (multi-class target)

# Encode fighter names with OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
all_fighters = pd.DataFrame(pd.concat([df2['fighter_A'], df2['fighter_B']], ignore_index=True), columns=['fighter'])
encoder.fit(all_fighters)

fighter_A_encoded = encoder.transform(df2[['fighter_A']].rename(columns={'fighter_A':'fighter'}))
fighter_B_encoded = encoder.transform(df2[['fighter_B']].rename(columns={'fighter_B':'fighter'}))

fighter_A_df = pd.DataFrame(fighter_A_encoded, columns=[f'A_{cat}' for cat in encoder.categories_[0]])
fighter_B_df = pd.DataFrame(fighter_B_encoded, columns=[f'B_{cat}' for cat in encoder.categories_[0]])


# Combine all features (fight stats + encoded fighters)
print(len(fighter_A_df))
print(len(fighter_B_df))
X = pd.concat([df2.drop(columns=['fighter_A', 'fighter_B', 'Winner', 'Winner_color']), fighter_A_df, fighter_B_df], axis=1)

print(1)
# Encode target labels numerically
label_encoder = LabelEncoder()
print(2)
y = label_encoder.fit_transform(df2['Winner_color'])

print(3)
# Train/test split
print(len(X))
print(len(y))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(4)
# Remove test samples with labels not in train (safe and simple):
train_labels = set(y_train)
mask = np.isin(y_test, list(train_labels))
X_test_filtered = X_test[mask]
y_test_filtered = y_test[mask]

print(5)

print(f"Removed {len(y_test) - len(y_test_filtered)} test samples with unseen labels.")







# #Model & Metric Limitations
# # Some metrics like classification_report require you to pass target_names that exactly match the unique classes.

# # If the number of unique target names is huge or mismatched, you get errors like:

# # arduino
# # Copy
# # Edit
# # ValueError: Number of classes, 62, does not match size of target_names, 165.

# # This error means your classification report is confused because the number of unique classes in your predictions (62) doesn't match the length of the target_names list (165).

# # Likely causes:

# # You passed a target_names list with 165 names, but your model predicted only 62 unique classes.

# # Your labels or target_names are mismatched or inconsistent.
# | Term              | What it Means                         | Example                               |
# | ----------------- | ------------------------------------- | ------------------------------------- |
# | **Classes**       | Unique values your model predicts     | `['Red', 'Blue', 'Draw']`             |
# | **Targets**       | Actual values in `y_test` and `preds` | `['Red', 'Red', 'Blue', 'Draw', ...]` |
# | **target\_names** | Names corresponding to each class     | `['Red', 'Blue', 'Draw']`             |

#     # target names are the unique values of Y
#     #classes are the unique values of Y_test
#     #targets are the values of y_test and predictions

#     | Term           | Meaning                                           |
# | -------------- | ------------------------------------------------- |
# | `y`            | The full label array (encoded)                    |
# | `y_train`      | Labels used to train model                        |
# | `y_test`       | Labels used to test model                         |
# | `preds`        | Model's predicted labels                          |
# | `classes`      | Unique labels in the full dataset (`le.classes_`) |
# | `targets`      | Actual labels like `y_test` or `preds`            |
# | `target_names` | Names that match the numeric classes              |



382
382
1
2
3
745
382


ValueError: Found input variables with inconsistent numbers of samples: [745, 382]

In [4]:
# step 3 mmodel fit

from catboost import CatBoostClassifier, Pool
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Identify categorical columns (by name or index)
categorical_features = []  # example columns


# Create and train CatBoost model
model = CatBoostClassifier(
    iterations=500,          # Number of boosting rounds
    learning_rate=0.05,      # Step size
    depth=8,                 # Tree depth
    eval_metric='Accuracy',  # Evaluation metric
    random_seed=42,
    verbose=100              # Print progress every 100 iterations
)

model.fit(
    X_train, y_train,
    cat_features=categorical_features,
    eval_set=(X_test_filtered, y_test_filtered),
    use_best_model=True
)


0:	learn: 0.9467009	test: 0.9565217	best: 0.9565217 (0)	total: 383ms	remaining: 3m 10s
100:	learn: 0.9465499	test: 0.9583333	best: 0.9583333 (1)	total: 20.4s	remaining: 1m 20s
200:	learn: 0.9465499	test: 0.9583333	best: 0.9583333 (1)	total: 39.1s	remaining: 58.1s
300:	learn: 0.9465499	test: 0.9583333	best: 0.9583333 (1)	total: 57.6s	remaining: 38.1s
400:	learn: 0.9465499	test: 0.9583333	best: 0.9583333 (1)	total: 1m 16s	remaining: 18.8s
499:	learn: 0.9465499	test: 0.9583333	best: 0.9583333 (1)	total: 1m 35s	remaining: 0us

bestTest = 0.9583333333
bestIteration = 1

Shrink model to first 2 iterations.


In [5]:
# Get the labels that are actually in y_test step 4 output prediction
unique_labels = np.unique(y_test_filtered)

# Get the correct names for those labels
target_names = label_encoder.inverse_transform(unique_labels)

# Predict and evaluate
y_pred = model.predict(X_test_filtered)
print("Accuracy:", accuracy_score(y_test_filtered, y_pred))
print(classification_report(y_test_filtered, y_pred, labels=unique_labels, target_names=target_names))

Accuracy: 0.9583333333333334
              precision    recall  f1-score   support

         red       0.00      0.00      0.00        69
     unknown       0.96      1.00      0.98      1587

    accuracy                           0.96      1656
   macro avg       0.48      0.50      0.49      1656
weighted avg       0.92      0.96      0.94      1656



C:\Users\admin\anaconda3\envs\datascience\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\admin\anaconda3\envs\datascience\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\admin\anaconda3\envs\datascience\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

In [ ]:
#deploy
#streamlit doesnt work in jupyter
from ipywidgets import interact
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# Dummy training data
np.random.seed(42)
df = pd.DataFrame({
    'fighter_A_height': np.random.randint(160, 200, 100),
    'fighter_A_reach': np.random.randint(160, 210, 100),
    'fighter_A_wins': np.random.randint(0, 30, 100),
    'fighter_B_height': np.random.randint(160, 200, 100),
    'fighter_B_reach': np.random.randint(160, 210, 100),
    'fighter_B_wins': np.random.randint(0, 30, 100),
    'winner': np.random.choice(['Fighter A', 'Fighter B'], 100)
})

X = df.drop(columns=['winner'])
y = df['winner']

clf = RandomForestClassifier()
clf.fit(X, y)

def predict_fight(
    fighter_A_height, fighter_A_reach, fighter_A_wins,
    fighter_B_height, fighter_B_reach, fighter_B_wins
):
    input_df = pd.DataFrame([[
        fighter_A_height, fighter_A_reach, fighter_A_wins,
        fighter_B_height, fighter_B_reach, fighter_B_wins
    ]], columns=X.columns)
    
    pred = clf.predict(input_df)[0]
    print(f"🏆 Predicted Winner: {pred}")

interact(
    predict_fight,
    fighter_A_height=(160, 200),
    fighter_A_reach=(160, 210),
    fighter_A_wins=(0, 30),
    fighter_B_height=(160, 200),
    fighter_B_reach=(160, 210),
    fighter_B_wins=(0, 30),
)